# 06 · Full shot compilation (query-by-shot layer)

Joins everything known about a shot into one record: **experiment goal** (extracted from the session's mini-proposal — extractive, not LLM-generated), **logbook sentiment** (preferring per-shot notes over session fallback), and **objective success**. Every field is independently marked available / unavailable.

**Honest coverage picture** (anchored on the 3,718 shots that have real physics parameters): 3,331 with a success check · 1,231 with sentiment · 639 with goal text · **483 (13 %) with all three**. The gap is a scope mismatch — the logbook archive spans 2012–2026 while the HDF5 extraction is a narrower slice — not a pipeline bug.

In [ ]:
import os, json, csv
import pandas as pd
from bs4 import BeautifulSoup

full_logbook_path = "/path/to/big_d3d_data/foundation_model_text/shotsummary/raw"
param_df = pd.read_csv("/path/to/outputs/shot_parameters_v2.csv")
success_df = pd.read_csv("/path/to/outputs/shot_success_scores.csv")

sentiment_lookup = {}
with open("/path/to/outputs/shot_sentiment.csv", "r") as f:
    for row in csv.DictReader(f):
        shot_number = int(row["shot_number"])
        existing = sentiment_lookup.get(shot_number)
        if existing is None or (existing["source"] == "session_fallback" and row["source"] == "per_shot_notes"):
            sentiment_lookup[shot_number] = row

success_lookup = {int(r["shot_number"]): r for r in success_df.to_dict("records")}

session_folders = sorted(os.listdir(full_logbook_path))
shot_to_session = {}
for folder in session_folders:
    metadata_path = os.path.join(full_logbook_path, folder, "metadata.json")
    if not os.path.exists(metadata_path):
        continue
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    shot_range = metadata.get("shot_range", "")
    if " - " in shot_range:
        try:
            start, end = [int(p.strip()) for p in shot_range.split(" - ")]
            for shot in range(start, end + 1):
                shot_to_session[shot] = folder
        except ValueError:
            continue

print("Parameters:", len(param_df), "| Success scores:", len(success_df),
      "| Sentiment shots:", len(sentiment_lookup), "| Session-mapped shots:", len(shot_to_session))

In [ ]:
def get_shot_goal(shot_number, shot_to_session, logbook_path):
    """
    Pulls the experiment's stated goal for a shot's session from the
    miniproposal. This is extractive (title + opening abstract text),
    not an LLM-generated summary
    """
    
    session_folder = shot_to_session.get(shot_number)
    if not session_folder:
        return None, None

    metadata_path = os.path.join(logbook_path, session_folder, "metadata.json")
    title = None
    if os.path.exists(metadata_path):
        with open(metadata_path, "r") as f:
            title = json.load(f).get("title")

    html_path = os.path.join(logbook_path, session_folder, "miniproposal.html")
    if not os.path.exists(html_path):
        return title, None

    with open(html_path, "r") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    text = soup.get_text(separator="\n", strip=True)

    # grab the first substantial paragraph (>150 chars) as the goal statement
    for para in text.split("\n"):
        if len(para.strip()) > 150:
            return title, para.strip()

    return title, None

print("Goal extraction function defined")

In [ ]:
def get_shot_summary(shot_number):
    """
    User-facing lookup: given a shot number, returns what's known about
    it - experiment goal (per session), sentiment, and objective target-vs-actual success.
    Every field is independently marked available/unavailable
    """
    result = {"shot_number": shot_number}

    param_row = param_df[param_df["shot_number"] == shot_number]
    result["has_parameters"] = len(param_row) > 0

    title, goal_text = get_shot_goal(shot_number, shot_to_session, full_logbook_path)
    result["session_title"] = title
    result["goal"] = goal_text
    result["has_goal"] = goal_text is not None

    sentiment_row = sentiment_lookup.get(shot_number)
    if sentiment_row:
        result["sentiment"] = sentiment_row["sentiment"]
        result["sentiment_source"] = sentiment_row["source"]
    else:
        result["sentiment"] = None
        result["sentiment_source"] = None
    result["has_sentiment"] = sentiment_row is not None

    success_row = success_lookup.get(shot_number)
    if success_row:
        result["overall_success"] = success_row.get("overall_success")
        result["ip_percent_diff"] = success_row.get("ip_percent_diff")
        result["nbi_percent_diff"] = success_row.get("nbi_percent_diff")
    else:
        result["overall_success"] = None
        result["ip_percent_diff"] = None
        result["nbi_percent_diff"] = None
    result["has_success_check"] = success_row is not None

    return result


def print_shot_summary(shot_number):
    r = get_shot_summary(shot_number)
    print(f"Shot {r['shot_number']}")
    print(f"  Session: {r['session_title']}")
    print(f"  Goal: {r['goal'][:200] + '...' if r['goal'] else 'not available'}")
    print(f"  Sentiment: {r['sentiment']} (source: {r['sentiment_source']})" if r['has_sentiment'] else "  Sentiment: not available")
    print(f"  Objective success: {r['overall_success']} (IP diff {r['ip_percent_diff']}%, NBI diff {r['nbi_percent_diff']}%)" if r['has_success_check'] else "  Success check: not available")

print("Query function defined")

In [ ]:
print_shot_summary(144229)

In [ ]:
all_shots = sorted(set(param_df["shot_number"]))
compiled = [get_shot_summary(s) for s in all_shots]

output_file = "/path/to/outputs/shot_full_compilation.csv"
with open(output_file, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(compiled[0].keys()))
    writer.writeheader()
    writer.writerows(compiled)
print("Saved to:", output_file)

has_goal = sum(1 for r in compiled if r["has_goal"])
has_sent = sum(1 for r in compiled if r["has_sentiment"])
has_succ = sum(1 for r in compiled if r["has_success_check"])
has_all = sum(1 for r in compiled if r["has_goal"] and r["has_sentiment"] and r["has_success_check"])

print(f"Of {len(compiled)} shots (all with real physics parameters):")
print(f"  With goal text:    {has_goal}")
print(f"  With sentiment:    {has_sent}")
print(f"  With success check:{has_succ}")
print(f"  With ALL THREE:    {has_all} ({100*has_all/len(compiled):.1f}%)")